# Chapter 4 (optional) — Normalisation and Pigment Abundance Maps

This notebook robustly rescales the four indices to a common `[0, 1]`
range, calculates relative pigment-proxy contributions, creates a pigment
composite, supports interactive thresholding, and overlays the result on a
geographic basemap.

**Important:** percentile normalisation and relative contributions are
scene-specific. They do not represent absolute pigment concentrations.

## 1. Load libraries and shared workshop functions

In [ ]:
from pathlib import Path

import folium
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import rasterio
from IPython.display import Markdown, clear_output, display
from matplotlib.colors import LinearSegmentedColormap, LogNorm, PowerNorm
from plotly.subplots import make_subplots
from rasterio.enums import ColorInterp
from rasterio.transform import Affine
from rasterio.warp import transform_bounds
from scipy.ndimage import gaussian_filter, gaussian_filter1d

# workshop_utils.py sits beside this notebook and holds the shared helpers.
from workshop_utils import (
    INDEX_INFO, load_data, compute_indices, pstretch, to_uint8,
    projected_plot_context, make_projected_heatmap, make_projected_image,
    apply_projected_axes,
)

pio.renderers.default = "plotly_mimetype+notebook_connected"

# This chapter uses only these four indices.
ANALYSIS_INDICES = ["NDVI", "CIRE", "ARI2", "CRI2"]
OUTPUT_DIR = Path("outputs")

## 2. Load data and calculate indices

The imagery is read straight from the workshop STAC catalogue, so there is
nothing to download first. This chapter uses four indices calculated from the
available MicaSense RedEdge-P Dual bands:

| Index | Equation | Role |
|---|---|---|
| NDVI | $(R_{842}-R_{668})/(R_{842}+R_{668})$ | Vegetation mask/reference |
| CIRE | $(R_{842}/R_{705})-1$ | Chlorophyll proxy |
| ARI2 | $R_{842}[(1/R_{560})-(1/R_{705})]$ | Anthocyanin proxy |
| CRI2 | $(1/R_{531})-(1/R_{705})$ | Carotenoid proxy |

The ARI2 and CRI2 equations use the nearest available sensor bands to the
nominal wavelengths used by these indices.

In [ ]:
# Read the study area from the STAC catalogue (takes 10-20 seconds).
ds = compute_indices(load_data(), names=ANALYSIS_INDICES)
ds

## 2.5 NDVI vegetation mask

A single hard-coded threshold controls which pixels are included in **all**
downstream analysis. Non-vegetation pixels are set to `NaN` in every index
immediately, so every later cell (normalisation, composites, pair plots,
export) automatically operates on the masked scene only.

In [ ]:
NDVI_THRESHOLD = 0.30   # pixels with NDVI <= threshold become NaN everywhere

veg_mask = ds["NDVI"] > NDVI_THRESHOLD   # NaN pixels count as non-vegetation
total_px, kept_px = veg_mask.size, int(veg_mask.sum())
print(f"NDVI threshold    : {NDVI_THRESHOLD}")
print(f"Total pixels      : {total_px:,}")
print(f"Vegetation pixels : {kept_px:,}  ({100 * kept_px / total_px:.1f} %)")
print(f"Masked-out pixels : {total_px - kept_px:,}  ({100 * (total_px - kept_px) / total_px:.1f} %)")

# Apply the mask to every index in the dataset.
ds.update(ds[ANALYSIS_INDICES].where(veg_mask))
print("\nNDVI mask applied to:", ", ".join(ANALYSIS_INDICES))

### Index distributions after masking

Each histogram marks the minimum (red), maximum (green) and median (black) of
the vegetation pixels.

In [ ]:
fig = make_subplots(rows=2, cols=2, subplot_titles=ANALYSIS_INDICES,
                    horizontal_spacing=0.12, vertical_spacing=0.12)
colours = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

for i, name in enumerate(ANALYSIS_INDICES):
    row, col = i // 2 + 1, i % 2 + 1
    vals = ds[name].values[np.isfinite(ds[name].values)]   # vegetation pixels only
    fig.add_trace(go.Histogram(x=vals, nbinsx=150, marker_color=colours[i], name=name),
                  row=row, col=col)
    for label, value, dash, colour in [
        ("min", np.nanmin(vals), "dash", "red"),
        ("max", np.nanmax(vals), "dash", "green"),
        ("median", np.nanmedian(vals), "solid", "black"),
    ]:
        fig.add_vline(x=float(value), line_dash=dash, line_color=colour, line_width=1.5,
                      annotation_text=f"{label}: {value:.3g}", annotation_position="top",
                      row=row, col=col)

fig.update_xaxes(title_text="Value")
fig.update_yaxes(title_text="Frequency", col=1)
fig.update_layout(height=700, width=1200, showlegend=False, title_x=0.5,
                  title_text=f"Index histograms — vegetation pixels only (NDVI > {NDVI_THRESHOLD})")
fig.show()

## 3. Robustly normalise each index to [0, 1]

Each index is scaled using its 2nd and 98th percentiles:

$$x' = \operatorname{clip}\left(\frac{x-P_2}{P_{98}-P_2},0,1\right).$$

The pigment proxies are then divided by their pixel-wise total to obtain
relative contributions.

### Calculate pixel-wise relative pigment-proxy contributions

Let the robust-normalised proxies be $A$ for `ARI2`, $C_h$ for `CIRE`, and $C_a$ for `CRI2`. The relative components are:

$$A_{rel}=\frac{A}{A+C_h+C_a},\qquad
C_{h,rel}=\frac{C_h}{A+C_h+C_a},\qquad
C_{a,rel}=\frac{C_a}{A+C_h+C_a}.$$

Pixels with missing inputs or a near-zero total are retained as NoData. The
NDVI mask from Section 2.5 already limits maps and exports to vegetation.

In [ ]:
EPS = 1e-6


def robust_normalise(array, qlow=2, qhigh=98):
    # P2-P98 normalisation to [0, 1]; invalid pixels stay NaN.
    low, high = np.nanpercentile(array, [qlow, qhigh])
    if high - low <= EPS:
        return np.where(np.isfinite(array), 0.0, np.nan).astype("float32")
    return np.clip((array - low) / (high - low), 0, 1).astype("float32")


normed, rows = {}, []
for name in ANALYSIS_INDICES:
    raw = ds[name].values
    normed[name] = robust_normalise(raw)
    p2, p98 = np.nanpercentile(raw, [2, 98])
    rows.append({
        "Index": name,
        "Raw min": np.nanmin(raw), "P2": p2, "P98": p98, "Raw max": np.nanmax(raw),
        "Normalised min": np.nanmin(normed[name]),
        "Normalised max": np.nanmax(normed[name]),
        "Normalised mean": np.nanmean(normed[name]),
    })
display(pd.DataFrame(rows).round(4))

# Relative contributions, in composite channel order R, G, B.
PIGMENTS = {
    "anth_rel": ("ARI2", "anthocyanin"),
    "cab_rel": ("CIRE", "chlorophyll"),
    "ccar_rel": ("CRI2", "carotenoid"),
}
pigment_total = sum(normed[index] for index, _ in PIGMENTS.values())
with np.errstate(divide="ignore", invalid="ignore"):
    for var, (index, pigment) in PIGMENTS.items():
        rel = np.where(pigment_total > EPS, normed[index] / pigment_total, np.nan)
        ds[var] = (("y", "x"), rel.astype("float32"))
        ds[var].attrs.update(long_name=f"Relative {pigment} proxy contribution",
                             source_index=index)

relative_sum = sum(ds[var] for var in PIGMENTS)
print(f"Relative-component sum: min={float(relative_sum.min()):.6f}, "
      f"mean={float(relative_sum.mean()):.6f}, max={float(relative_sum.max()):.6f}")

## 4. Create the pigment composite

- Red = ARI2 relative contribution (anthocyanin proxy)
- Green = CIRE relative contribution (chlorophyll proxy)
- Blue = CRI2 relative contribution (carotenoid proxy)

The next cell builds the arrays reused by every map below: the pigment
composite (with a transparency band for masked pixels) and a percentile-stretched
true-colour reference (R = 668 nm, G = 560 nm, B = 475 nm).

In [ ]:
PIGMENT_LEGEND = "Anthocyanin (ARI2)=R, Chlorophyll (CIRE)=G, Carotenoid (CRI2)=B"

pigment_rgb = np.dstack([ds[var].values for var in PIGMENTS])   # (rows, cols, 3), NaN = masked
pigment_valid = np.isfinite(pigment_rgb).all(axis=2)
pigment_uint8 = to_uint8(pigment_rgb)
pigment_rgba = np.dstack([pigment_uint8, np.where(pigment_valid, 255, 0).astype(np.uint8)])

rgb_true = np.dstack([pstretch(ds[band].values) for band in ("red668", "green560", "blue")])
context = projected_plot_context(ds)
unit = context["unit_label"]

fig = go.Figure(make_projected_image(
    ds, pigment_rgba, context=context,
    customdata=pigment_rgb.round(3),   # hover values
    hovertemplate=(
        f"Easting: %{{x:,.3f}} {unit}<br>"
        f"Northing: %{{y:,.3f}} {unit}<br>"
        "ARI2 relative abundance: %{customdata[0]:.3f}<br>"
        "CIRE relative abundance: %{customdata[1]:.3f}<br>"
        "CRI2 relative abundance: %{customdata[2]:.3f}"
        "<extra></extra>"
    ),
))
apply_projected_axes(fig, context)
fig.update_layout(
    title=dict(text="Relative Pigment-Abundance Map"
                    "<br><sup>Red = ARI2 · Green = CIRE · Blue = CRI2</sup>",
               x=0.5, xanchor="center"),
    template="plotly_white", width=750, height=650,
    margin=dict(l=90, r=40, t=100, b=80),
)
fig.show(config={"scrollZoom": True, "displaylogo": False})

The same composite beside the true-colour image, with masked pixels shown in white:

In [ ]:
pigment_white = np.where(pigment_valid[..., None], np.clip(pigment_rgb, 0, 1), 1.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
panels = [
    (rgb_true, "True RGB"),
    (pigment_white, f"Relative Pigment Composite — NDVI > {NDVI_THRESHOLD:.2f}\n{PIGMENT_LEGEND}"),
]
for ax, (image, title) in zip(axes, panels):
    ax.imshow(image, interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")
plt.show()

And draped at 50% opacity over the true-colour image:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(rgb_true, interpolation="nearest")
ax.imshow(np.nan_to_num(pigment_rgb), alpha=np.where(pigment_valid, 0.5, 0.0),
          interpolation="nearest")
ax.set_title(f"Relative Pigment Composite over True RGB\nNDVI > {NDVI_THRESHOLD:.2f}; {PIGMENT_LEGEND}")
ax.axis("off")
plt.tight_layout()
plt.show()

Finally, a linked three-panel view: zoom or pan one panel and the others follow.

In [ ]:
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.04,
                    subplot_titles=["RGB true-colour", "NDVI (normalised)",
                                    "Relative pigment abundance"])
fig.add_trace(make_projected_image(ds, to_uint8(rgb_true), context=context), row=1, col=1)
fig.add_trace(make_projected_heatmap(ds, array=normed["NDVI"], context=context,
                                     colorscale="YlGn", zmin=0, zmax=1,
                                     value_label="NDVI (normalised)"), row=1, col=2)
fig.add_trace(make_projected_image(ds, pigment_uint8, context=context), row=1, col=3)

for col in (1, 2, 3):
    apply_projected_axes(fig, context, row=1, col=col, axis_number=col, y_title=col == 1)
    if col > 1:
        fig.update_xaxes(matches="x", row=1, col=col)
        fig.update_yaxes(matches="y", row=1, col=col)

fig.update_layout(
    title=dict(text="RGB Reference, NDVI and Relative Pigment Abundance"
                    "<br><sup>Pigment composite: Red = ARI2 · Green = CIRE · Blue = CRI2</sup>",
               x=0.5, xanchor="center"),
    template="plotly_white", width=1300, height=600, showlegend=False,
    margin=dict(l=90, r=30, t=105, b=85),
)
fig.show(config={"scrollZoom": True, "displaylogo": False,
                 "modeBarButtonsToAdd": ["drawrect", "eraseshape"]})

## 5. Masked-data threshold explorer

The NDVI mask has already been applied. The explorer below shows
how a **secondary threshold** on the normalised index further
subsets the already-masked vegetation pixels.

In [ ]:
th_idx = widgets.Dropdown(options=ANALYSIS_INDICES, value="NDVI", description="Index:")
th_val = widgets.FloatSlider(value=0.20, min=0, max=1, step=0.01, description="Threshold:",
                             readout_format=".2f")
threshold_output = widgets.Output()


def render_threshold(change=None):
    with threshold_output:
        clear_output(wait=True)
        name, info = th_idx.value, INDEX_INFO[th_idx.value]
        valid = np.isfinite(normed[name])
        selected = valid & (normed[name] >= th_val.value)
        kept, total = int(selected.sum()), int(valid.sum())

        fig = go.Figure(make_projected_heatmap(
            ds, array=np.where(selected, normed[name], np.nan), context=context,
            colorscale=info["cmap"], zmin=0, zmax=1, showscale=True,
            colorbar=dict(title=f"{name} (norm.)", thickness=14),
            value_label=f"{name} (normalised)",
        ))
        apply_projected_axes(fig, context)
        fig.update_layout(
            title=f"{name} ≥ {th_val.value:.2f} — {kept:,} of {total:,} valid pixels "
                  f"({100 * kept / max(total, 1):.1f}%)",
            template="plotly_white", width=850, height=650,
            margin=dict(l=90, r=100, t=80, b=80),
        )
        fig.show(config={"scrollZoom": True, "displaylogo": False})
        display(Markdown(f"**{name}:** {info['full_name']}  \n"
                         f"**Interpretation:** {info['interpretation']}"))


for control in (th_idx, th_val):
    control.observe(render_threshold, names="value")
display(widgets.HBox([th_idx, th_val]), threshold_output)
render_threshold()

The second explorer keeps normalised NDVI on the left and applies its
threshold to both panels, so you can see what a pigment proxy looks like at
the NDVI-selected pixels.

In [ ]:
comparison_index = widgets.Dropdown(options=ANALYSIS_INDICES[1:], value="CIRE",
                                    description="Right index:")
ndvi_threshold = widgets.FloatSlider(value=0.20, min=0, max=1, step=0.01,
                                     description="NDVI threshold:", readout_format=".2f")
comparison_output = widgets.Output()


def render_threshold_comparison(change=None):
    with comparison_output:
        clear_output(wait=True)
        right, threshold = comparison_index.value, ndvi_threshold.value
        valid = np.isfinite(normed["NDVI"])
        selected = valid & (normed["NDVI"] >= threshold)
        kept, total = int(selected.sum()), int(valid.sum())

        fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                            subplot_titles=[f"NDVI ≥ {threshold:.2f}",
                                            f"{right} at NDVI-selected pixels"])
        for col, (name, colorbar_x) in enumerate([("NDVI", 0.455), (right, 1.01)], start=1):
            fig.add_trace(make_projected_heatmap(
                ds, array=np.where(selected, normed[name], np.nan), context=context,
                colorscale=INDEX_INFO[name]["cmap"], zmin=0, zmax=1, showscale=True,
                colorbar=dict(title=name, x=colorbar_x, xanchor="left", len=0.82, thickness=12),
                value_label=f"{name} (normalised)",
            ), row=1, col=col)
            apply_projected_axes(fig, context, row=1, col=col, axis_number=col)
        fig.update_xaxes(matches="x", row=1, col=2)
        fig.update_yaxes(matches="y", row=1, col=2)

        fig.update_layout(
            title=dict(text=f"NDVI ≥ {threshold:.2f}: {kept:,} of {total:,} valid pixels "
                            f"({100 * kept / max(total, 1):.1f}%)",
                       x=0.5, xanchor="center"),
            template="plotly_white", width=1200, height=600, showlegend=False,
            margin=dict(l=90, r=110, t=100, b=80),
        )
        fig.show(config={"scrollZoom": True, "displaylogo": False})
        info = INDEX_INFO[right]
        display(Markdown(f"**Right panel: {right} — {info['full_name']}**  \n"
                         f"**Interpretation:** {info['interpretation']}"))


for control in (comparison_index, ndvi_threshold):
    control.observe(render_threshold_comparison, names="value")
display(widgets.HBox([comparison_index, ndvi_threshold]), comparison_output)
render_threshold_comparison()

## 6. Pigment abundance composite on a basemap

Masked pixels are transparent, so the satellite or street basemap shows
through. Use the layer control to switch basemaps or hide the composite.

In [ ]:
# Corners of the study area in longitude/latitude for the web map.
(x_min, x_max), (y_min, y_max) = context["x_range"], context["y_range"]
west, south, east, north = transform_bounds(ds.attrs["crs"], "EPSG:4326",
                                            x_min, y_min, x_max, y_max, densify_pts=21)
bounds = [[south, west], [north, east]]

# Downsample to roughly 500 pixels across to keep the notebook light.
step = max(1, max(pigment_rgba.shape[:2]) // 500)

pigment_map = folium.Map(location=[(south + north) / 2, (west + east) / 2],
                         tiles=None, zoom_start=17, control_scale=True)
folium.TileLayer("openstreetmap", name="OpenStreetMap").add_to(pigment_map)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery", name="Esri Satellite",
).add_to(pigment_map)
folium.raster_layers.ImageOverlay(
    image=pigment_rgba[::step, ::step], bounds=bounds, opacity=0.88, interactive=True,
    name="Relative pigments (R=ARI2 anthocyanin, G=CIRE chlorophyll, B=CRI2 carotenoid)",
).add_to(pigment_map)
folium.LayerControl(collapsed=False).add_to(pigment_map)
pigment_map.fit_bounds(bounds)
pigment_map

## 7. Pairwise relationships between the indices

Both figures below are lower-triangle pair plots of the four (masked) indices.
The diagonal panels show each index's smoothed distribution; each lower panel
shows the joint distribution of two indices, labelled with its Pearson
correlation coefficient $r$.

- The first figure keeps only pixels inside every index's P2–P98 range and
  draws the smoothed joint pixel density.
- The second uses every valid pixel, drawn as points coloured by how crowded
  their neighbourhood is.

In [ ]:
# One row per pixel valid in all four indices, one column per index.
pair_data = np.column_stack([ds[name].values.ravel() for name in ANALYSIS_INDICES])
pair_data = pair_data[np.isfinite(pair_data).all(axis=1)].astype("float64")


def pair_plot(data, lower_panel, colours, title, figsize=(10, 10)):
    # Diagonal: smoothed histogram. Lower triangle: lower_panel(ax, x, y) plus r.
    fill, line = colours
    n = len(ANALYSIS_INDICES)
    fig, axes = plt.subplots(n, n, figsize=figsize, constrained_layout=True)
    for row in range(n):
        for col in range(n):
            ax = axes[row, col]
            if col > row:
                ax.remove()
                continue
            x = data[:, col]
            if row == col:
                density, edges = np.histogram(x, bins=100, density=True)
                centres = (edges[:-1] + edges[1:]) / 2
                smooth = gaussian_filter1d(density, sigma=1.3)
                ax.fill_between(centres, smooth, color=fill, alpha=0.7)
                ax.plot(centres, smooth, color=line, linewidth=1.0)
                ax.set_ylim(bottom=0)
                ax.set_yticks([])
            else:
                y = data[:, row]
                lower_panel(ax, x, y)
                ax.set_ylim(y.min(), y.max())
                ax.text(0.04, 0.95, f"r = {np.corrcoef(x, y)[0, 1]:.2f}",
                        transform=ax.transAxes, ha="left", va="top", fontsize=10, color=line,
                        bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=1.5))
            ax.set_xlim(x.min(), x.max())

            # Labels on the outer panels only.
            if row == n - 1:
                ax.set_xlabel(ANALYSIS_INDICES[col], fontsize=11)
            else:
                ax.tick_params(axis="x", labelbottom=False)
            if col == 0 and row > 0:
                ax.set_ylabel(ANALYSIS_INDICES[row], fontsize=11)
            elif col > 0:
                ax.tick_params(axis="y", labelleft=False)
            ax.tick_params(labelsize=8)
            for side in ("top", "right"):
                ax.spines[side].set_visible(False)
    fig.suptitle(title, fontsize=14)
    plt.show()

In [ ]:
# Figure 1: smoothed joint density of pixels within every index's P2-P98 range.
low, high = np.percentile(pair_data, [2, 98], axis=0)
trimmed = pair_data[((pair_data >= low) & (pair_data <= high)).all(axis=1)]
print(f"Pixels used: {len(trimmed):,} (P2–P98)")
display(pd.DataFrame(np.corrcoef(trimmed, rowvar=False),
                     index=ANALYSIS_INDICES, columns=ANALYSIS_INDICES).round(3))

orange_density = LinearSegmentedColormap.from_list(
    "density_orange", ["#ffffff", "#fee8dc", "#fdbb84", "#ef6548", "#990000"])


def smoothed_density(ax, x, y):
    counts, x_edges, y_edges = np.histogram2d(x, y, bins=150)
    counts = gaussian_filter(counts, sigma=1.3)   # KDE-like smoothing
    ax.pcolormesh(x_edges, y_edges, np.ma.masked_less_equal(counts.T, 0),
                  cmap=orange_density, norm=PowerNorm(gamma=0.45, vmin=0, vmax=counts.max()),
                  rasterized=True)


pair_plot(trimmed, smoothed_density, colours=("#d8b9a5", "#b2182b"),
          title=f"Spectral-Index Pairwise Density and Pearson Correlation\n"
                f"P2–P98; n={len(trimmed):,} pixels")

In [ ]:
# Figure 2: every valid pixel, coloured by the pixel count of its local 2-D histogram cell.
blue_density = LinearSegmentedColormap.from_list(
    "blue_density", ["#f7fbff", "#deebf7", "#c6dbef", "#9ecae1", "#6baed6", "#4292c6", "#2166ac"])


def density_scatter(ax, x, y, bins=180):
    counts, x_edges, y_edges = np.histogram2d(x, y, bins=bins)
    x_bin = np.clip(np.searchsorted(x_edges, x, side="right") - 1, 0, bins - 1)
    y_bin = np.clip(np.searchsorted(y_edges, y, side="right") - 1, 0, bins - 1)
    density = counts[x_bin, y_bin]
    order = np.argsort(density)   # draw dense points last
    ax.scatter(x[order], y[order], c=density[order], cmap=blue_density, s=0.3,
               norm=LogNorm(vmin=1, vmax=max(2.0, np.percentile(density, 99.5)), clip=True),
               linewidths=0, rasterized=True)


pair_plot(pair_data, density_scatter, colours=("#c6dbef", "#2166ac"), figsize=(9, 9),
          title=f"Spectral-Index Pairwise Scatter, Coloured by Point Density\n"
                f"All valid pixels; n={len(pair_data):,}")

## 8. Save the pigment composite

The three relative-contribution bands are written to a float32 GeoTIFF with
the same CRS and pixel grid as the input imagery. Masked pixels are stored as
NoData (`-9999`).

In [ ]:
out_path = OUTPUT_DIR / "pigment_rgb_relative.tif"
out_path.parent.mkdir(parents=True, exist_ok=True)

NODATA = -9999.0
CHANNELS = [("anthocyanin", "ARI2", "Red"), ("chlorophyll", "CIRE", "Green"),
            ("carotenoid", "CRI2", "Blue")]

# Rasterio expects (bands, rows, columns).
export = np.where(pigment_valid[..., None], pigment_rgb, NODATA)
export = np.moveaxis(export, -1, 0).astype("float32")

profile = {
    "driver": "GTiff", "height": export.shape[1], "width": export.shape[2],
    "count": 3, "dtype": "float32", "nodata": NODATA,
    "crs": ds.attrs["crs"], "transform": Affine(*ds.attrs["transform"][:6]),
    "compress": "deflate", "predictor": 3, "tiled": True, "BIGTIFF": "IF_SAFER",
}
with rasterio.open(out_path, "w", **profile) as dst:
    dst.write(export)
    dst.colorinterp = (ColorInterp.red, ColorInterp.green, ColorInterp.blue)
    for band, (pigment, index, colour) in enumerate(CHANNELS, start=1):
        dst.set_band_description(band, f"Relative {pigment} — {index} — {colour}")
    dst.write_mask(pigment_valid.astype(np.uint8) * 255)
    dst.update_tags(
        product="Relative pigment RGB composite",
        red_channel="ARI2 relative anthocyanin",
        green_channel="CIRE relative chlorophyll",
        blue_channel="CRI2 relative carotenoid",
        normalisation="P2-P98 scaling followed by pixel-wise relative contribution",
        ndvi_threshold=str(NDVI_THRESHOLD),
    )

# Read the file back to check it.
with rasterio.open(out_path) as src:
    print(f"Saved: {out_path}")
    print("Shape:", (src.count, src.height, src.width))
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Descriptions:", src.descriptions)
    for band, (pigment, index, _) in zip(src.read(masked=True), CHANNELS):
        print(f"{pigment.capitalize()}/{index}: min={band.min():.4f}, max={band.max():.4f}")

### Reading the pigment composite

The three channels are relative contributions and sum to approximately 1
for each valid pixel. Therefore, colour represents relative composition,
not total pigment concentration.

| Colour | Dominant relative signal | Interpretation |
|---|---|---|
| Red | High ARI2 contribution | Anthocyanin proxy dominates |
| Green | High CIRE contribution | Chlorophyll proxy dominates |
| Blue | High CRI2 contribution | Carotenoid proxy dominates |
| Yellow | ARI2 + CIRE | Anthocyanin and chlorophyll proxies are jointly strong |
| Cyan | CIRE + CRI2 | Chlorophyll and carotenoid proxies are jointly strong |
| Magenta | ARI2 + CRI2 | Anthocyanin and carotenoid proxies are jointly strong |
| White/transparent | Masked | Non-vegetation or invalid input |

## Activities

1. For CIRE, calculate the P2-P98 normalised value of a raw value by hand.
2. Select ARI2 in the threshold explorer. At what threshold does the
   selected area drop below 20%? Are the retained pixels clustered?
3. Compare blue, green, and red-dominant areas against the satellite layer.

The composite is a relative visual summary, not an estimate of absolute
biochemical concentration. Interpret patterns alongside field observations
and environmental context.